<p style="text-align:center">
    <a href="https://skills.network" target="_blank">
    <img src="https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/assets/logos/SN_web_lightmode.png" width="200" alt="Skills Network Logo"  />
    </a>
</p>


# **Imputing Missing Values | Preparing Survey Data for Analysis**
**IBM Skills Network Guided Lab | Personalized Portfolio Edition**


### Project Overview
Survey participants do not always answer every question. When responses are missing, analysts need to determine whether to leave them blank, exclude them, or estimate a replacement. Each choice can affect the conclusions drawn from the data.

In this guided project, I examine missing responses in a developer survey dataset, remove exact duplicate rows, and demonstrate mode imputation on the RemoteWork field. I also inspect compensation-related columns to identify data-quality concerns that require further investigation.

**Business question:** How can missing survey responses be handled consistently, and what trade-offs should be considered before using the cleaned data for workforce analysis?

**Workflow:** Load dataset → Identify and remove duplicate rows → Measure missing values → Examine remote-work categories → Impute missing responses → Inspect compensation distributions.

**Skills demonstrated:** Python, Pandas, duplicate detection, missing-value analysis, categorical mode imputation, descriptive statistics, and data-quality interpretation.

**Attribution:** IBM Skills Network guided lab with added portfolio explanations and interpretation. Original code and attribution are retained below.

Estimated time needed: **30** minutes


## Objectives


In this lab, you will perform the following:
-   Identify missing values in the dataset.
-   Apply techniques to impute missing values in the dataset.
-   Use suitable techniques to normalize data in the dataset.

-----


#### Install needed library


In [1]:
!pip install pandas

### Step 1: Import Required Libraries


In [12]:
import pandas as pd

### Step 2: Load the Dataset Into a Dataframe


##### **Read Data**
**Why start by loading and previewing the data?**
The dataset contains survey responses from developers. A quick preview helps verify that the CSV loaded and introduces the fields used later, including RemoteWork, CompTotal, and ConvertedCompYearly.

**Reader's guide:** Each row represents one response; each column represents a survey question or recorded attribute. A blank value (NaN) means the response is missing in the dataset, not necessarily that the respondent made an error.

In [13]:
file_path ="https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/UDKAZw-kz18Yj8P6icf_qw/survey-data-duplicates.csv"
df = pd.read_csv(file_path)

# Display the first few rows to ensure it loaded correctly
print(df.head())

   ResponseId                      MainBranch                 Age  \
0           1  I am a developer by profession  Under 18 years old   
1           2  I am a developer by profession     35-44 years old   
2           3  I am a developer by profession     45-54 years old   
3           4           I am learning to code     18-24 years old   
4           5  I am a developer by profession     18-24 years old   

            Employment RemoteWork   Check  \
0  Employed, full-time     Remote  Apples   
1  Employed, full-time     Remote  Apples   
2  Employed, full-time     Remote  Apples   
3   Student, full-time        NaN  Apples   
4   Student, full-time        NaN  Apples   

                                    CodingActivities  \
0                                              Hobby   
1  Hobby;Contribute to open-source projects;Other...   
2  Hobby;Contribute to open-source projects;Other...   
3                                                NaN   
4                                 

### Step 3. Finding and Removing Duplicates
##### Task 1: Identify duplicate rows in the dataset.


**Why check for duplicate rows?**
An identical row appearing more than once can inflate counts and distort percentages. The duplicated() method identifies repeated rows across all columns.

**Saved result:** The original dataset contains 10 exact duplicate rows. This check does not establish whether different rows might represent the same person; it only detects complete row matches.

In [14]:
## Write your code here
duplicate_rows = df.duplicated().sum()
print("Number of duplicate rows:", duplicate_rows)

Number of duplicate rows: 10

##### Task 2: Remove the duplicate rows from the dataframe.



**What happens when duplicates are removed?**
drop_duplicates() retains one copy of each identical row, while reset_index(drop=True) assigns a clean sequence of row labels. This step prepares the data for subsequent counts so repeated rows do not contribute twice. It does not automatically fix missing values or inconsistencies in other fields.

In [16]:
## Write your code here
df = df.drop_duplicates().reset_index(drop=True)

### Step 4: Finding Missing Values
##### Task 3: Find the missing values for all columns.


**Why inspect missing values across all columns?**
Different survey questions can have very different response rates. Counting nulls by column identifies fields that may need additional attention before comparisons or modeling.

In [17]:
## Write your code here
print(df.isnull().sum())

ResponseId                 0
MainBranch                 0
Age                        0
Employment                 0
RemoteWork             10631
                       ...  
JobSatPoints_11        35992
SurveyLength            9255
SurveyEase              9199
ConvertedCompYearly    42002
JobSat                 36311
Length: 114, dtype: int64

In the saved results, RemoteWork has 10,631 missing responses after duplicates are removed. ConvertedCompYearly has 42,002 missing responses and JobSat has 36,311. This illustrates why a cleaning strategy should be chosen separately for each field.

##### Task 4: Find out how many rows are missing in the column RemoteWork.


**Why isolate RemoteWork?**
The lab focuses on RemoteWork because it records categories rather than numerical measurements. We first establish the number of blanks so we can verify the effect of imputation afterward.

Before imputation: 10,631 missing entries in RemoteWork.

In [19]:
## Write your code here
missing_row_remote = df['RemoteWork'].isnull().sum()
print("Missing rows in RemoteWork:", missing_row_remote)


Missing rows in RemoteWork: 10631

### Step 5. Imputing Missing Values
##### Task 5: Find the value counts for the column RemoteWork.


In [20]:
## Write your code here
df['RemoteWork'].value_counts()

RemoteWork
Hybrid (some remote, some in-person)    23015
Remote                                  20831
In-person                               10960
Name: count, dtype: int64

These are counts of non-missing responses in the saved dataset. They should not be interpreted as population-wide employment rates.

##### Task 6: Identify the most frequent (majority) value in the RemoteWork column.



**What is mode imputation?**
The mode is the category that occurs most frequently. In this dataset, it is Hybrid (some remote, some in-person).

Using the mode to fill blanks is simple and preserves the field's categorical format. However, it assumes the most common observed answer is a reasonable replacement for every missing answer, an assumption that may not hold if nonresponse is systematic.

In [21]:
## Write your code here
freq_value = df['RemoteWork'].mode()[0]
print("Most frequent value is:", freq_value)

Most frequent value is: Hybrid (some remote, some in-person)

##### Task 7: Impute (replace) all the empty rows in the column RemoteWork with the majority value.



**How does imputation change the dataset?**
fillna(freq_value) replaces each missing RemoteWork entry with the observed mode.

**Saved result:** The number of missing RemoteWork entries falls from 10,631 to 0.

**Important interpretation:** The column is now complete, but the newly filled answers are estimated, not reported by respondents. Mode imputation increases the count of the Hybrid category and can bias comparisons between remote-work arrangements. Keep track of which entries were originally missing in a real analysis.

In [23]:
## Write your code here
df['RemoteWork'] = df['RemoteWork'].fillna(freq_value)
print("Remaining missing rows in RemoteWork:" , df['RemoteWork'].isnull().sum())


Remaining missing rows in RemoteWork: 0

##### Task 8: Check for any compensation-related columns and describe their distribution.



**Why inspect compensation separately?**
The notebook uses describe() to summarize CompTotal and ConvertedCompYearly. These numerical fields require more caution than a small set of work-arrangement categories.

The saved output shows a median ConvertedCompYearly of 65,000, with a much higher mean of approximately 86,155 and a maximum exceeding 16 million. This suggests a strongly skewed distribution or potential extreme values worth checking.

The CompTotal column also includes an exceptionally large value (1e150), making its mean unreliable as a description of a typical respondent. The notebook does not impute these compensation fields; it only examines their distributions. These fields should not be compared without understanding their definitions and data quality.

In [24]:
## Write your code here
df[['CompTotal', 'ConvertedCompYearly']].describe()

,CompTotal,ConvertedCompYearly
count,3.374000e+04,2.343500e+04
mean,2.963841e+145,8.615529e+04
std,5.444117e+147,1.867570e+05
min,0.000000e+00,1.000000e+00
25%,6.000000e+04,3.271200e+04
50%,1.100000e+05,6.500000e+04
75%,2.500000e+05,1.079715e+05
max,1.000000e+150,1.625660e+07


### Key Findings and Interpretation
- 10 exact duplicate rows were identified and removed before missing-value analysis.
- 10,631 RemoteWork responses were missing in the cleaned dataset.
- The most frequent observed response was Hybrid (some remote, some in-person).
- Mode imputation reduced missing RemoteWork values to zero, but the replacement answers remain estimates.
- Compensation summaries reveal extreme values and substantial missingness, requiring further validation before salary analysis.

### Limitations and Next Steps
This is a guided practice exercise, not a validated estimate of remote-work patterns or compensation in the wider workforce. The mode-based replacement may distort category proportions, especially if missing responses differ systematically from observed ones. The notebook's objectives mention normalization, but the included code does not implement a normalization method.

Useful next steps would be to calculate missing percentages, flag imputed records, compare category shares before and after imputation, investigate why RemoteWork is missing, and review compensation outliers and units before further analysis.

### Skills Demonstrated
- Data preparation: CSV loading, duplicate detection, and duplicate removal
- Missing-data analysis: isnull(), frequency tables, and missing counts
- Categorical imputation: mode() and fillna()
- Exploratory analysis: value_counts() and describe()
- Analytical communication: Explaining trade-offs and distinguishing observed from estimated values

**Portfolio note:** This is an IBM Skills Network guided lab with additional explanations. Findings are based on saved notebook outputs; the code has not been independently rerun.

<!--
## Change Log
|Date (YYYY-MM-DD)|Version|Changed By|Change Description|
|-|-|-|-|
|2024-11-05|1.3|Madhusudhan Moole|Updated lab|
|2024-10-29|1.2|Madhusudhan Moole|Updated lab|
|2024-09-27|1.1|Madhusudhan Moole|Updated lab|
|2024-09-26|1.0|Raghul Ramesh|Created lab|
--!>


Copyright © IBM Corporation. All rights reserved.
